In [48]:
import importlib
import Functions_library as flb
from pathlib import Path

flb = importlib.reload(flb)

In [41]:
notebook_dir = Path.cwd()
h5_dirs_path = notebook_dir.parent / "h5"

# Choose two folders from the same particle type and simulation category.
source_1_dir = h5_dirs_path / "MC_EGAM1"
source_2_dir = h5_dirs_path / "MC_EGAM7"


def data_category(path):
    name = path.name.upper()
    particle = "MUONS" if "MUON" in name else "EGAM" if "EGAM" in name else None
    simulation = "DATA" if name.startswith("DATA_") else "MC" if name.startswith("MC_") else None
    if particle is None or simulation is None:
        raise ValueError(
            f"Cannot determine particle/simulation category from folder name: {path.name}"
        )
    return particle, simulation


if data_category(source_1_dir) != data_category(source_2_dir):
    raise ValueError(
        "Cannot mix folders with different particle types or categories: "
        f"{source_1_dir.name} and {source_2_dir.name}"
    )

h5_files_1 = sorted(
    path for path in source_1_dir.iterdir() if path.is_file() and path.suffix == ".h5"
)
h5_files_2 = sorted(
    path for path in source_2_dir.iterdir() if path.is_file() and path.suffix == ".h5"
)

if not h5_files_1:
    raise FileNotFoundError(f"No files found in {source_1_dir}")
if not h5_files_2:
    raise FileNotFoundError(f"No files found in {source_2_dir}")

print(f"Mixing {source_1_dir.name} with {source_2_dir.name}")

Mixing MC_EGAM1 with MC_EGAM7


In [74]:
dataset = flb.H5EgammaDataset_fully_batched(
    files=h5_files_1,
    mix_files=h5_files_2,
    mixture_ratio=0.5,
    mixture_seed=42,
)
sampler = flb.ShuffledContiguousBatchSampler(
    len(dataset),
    batch_size=1024,
    dataset=dataset,
)
loader = flb.DataLoader(dataset, batch_sampler=sampler, num_workers=0)

In [75]:
x, y = next(iter(loader))

In [18]:
x.shape

torch.Size([1024, 6958])